# Notebook 14: ConvNeXt & Module 06 Capstone

**Module:** 06 CNN  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Understand modernized CNN design
2. Compare ConvNeXt with ViT preview
3. Complete MNIST CNN and CIFAR-10 projects
4. Connect to water-bodies encoder

## ConvNeXt (Liu et al., 2022)

Modernizes ResNet with ViT-inspired design choices:
- Larger kernels (7×7)
- LayerNorm instead of BatchNorm
- GELU instead of ReLU
- Inverted bottleneck (like Transformer FFN)

**Bridge to Module 10:** ConvNeXt borrows from Transformers; ViT borrows from CNNs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
classes = train_set.classes
print(f'CIFAR-10: {len(train_set)} train, {len(test_set)} test, {len(classes)} classes')

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total, correct, loss_sum = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        total += len(y)
    return loss_sum/total, correct/total

def evaluate(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            loss = criterion(out, y)
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum/total, correct/total

In [ ]:
class ConvNeXtBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, 7, padding=3, groups=dim)  # depthwise
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4*dim)  # pointwise
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4*dim, dim)
    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = x.permute(0,2,3,1); x = self.norm(x)
        x = self.pwconv2(self.act(self.pwconv1(x)))
        x = x.permute(0,3,1,2)
        return residual + x

block = ConvNeXtBlock(64)
print(f'ConvNeXt block: {block(torch.randn(1,64,16,16)).shape}')

## Module 06 Capstone: MNIST CNN

Beat your Module 05 MLP on MNIST with a simple CNN.

In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64*7*7, 128), nn.ReLU(), nn.Linear(128, 10),
        )
    def forward(self, x): return self.net(x)

print('MNIST CNN — train this to beat Module 05 MLP (~98% target)')

## water-bodies-detection Connection

```
Your pipeline:
  Input: (6, 512, 512) Planet bands
  Encoder: SE-ResNet50 (Module 06 ResNet + channel attention)
  Decoder: UNet++ (Module 07)
  Output: (2, 512, 512) aqua + boundary masks
```

Module 06 teaches what happens inside that ResNet50 encoder.

## Module 06 Complete

**Next:** Module 07 Segmentation — UNet, UNet++, and your water-bodies project.

---

## Summary

ConvNeXt modernizes CNNs. You now understand every major architecture from LeNet to ConvNeXt.